In [ ]:
from pyspark.sql.functions import col, to_date
bronze_taxi = spark.read.table("Bronze_Lakehouse.dbo.taxi_raw")

silver_taxi = (
    bronze_taxi
    .withColumn("trip_date", to_date("tpep_pickup_datetime"))
    .withColumn("fare_amount", col("fare_amount").cast("double"))
    .withColumn("total_amount", col("total_amount").cast("double"))
    .withColumn("trip_distance", col("trip_distance").cast("double"))
    .withColumn("passenger_count", col("passenger_count").cast("int"))
    .withColumn("PULocationID", col("PULocationID").cast("int"))
    .withColumn("DOLocationID", col("DOLocationID").cast("int"))
    
    .filter(col("PULocationID").isNotNull())
    .filter(col("total_amount").isNotNull())
    .filter(col("fare_amount") > 0)
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))

    #Over 120mhp
    .filter(col("trip_distance")<(col("tpep_dropoff_datetime").cast("long")-col("tpep_pickup_datetime").cast("long"))*0.033)
    .filter(col("fare_amount")<5000)
    .filter((col("tpep_dropoff_datetime").cast("long")-col("tpep_pickup_datetime").cast("long"))<86400)
    .dropDuplicates(["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime","PULocationID","DOLocationID"])
)
(
    silver_taxi.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("Silver_Lakehouse.dbo.taxi_silver")
)
print(f"Total rows: {silver_taxi.count():,}")

In [ ]:
%pip install geopandas

In [ ]:
import pandas as pd
import geopandas as gpd
from pyspark.sql.functions import col, to_timestamp, from_utc_timestamp, to_date

zones_url = "https://data.cityofnewyork.us/resource/8meu-9t5y.csv"
zones_pd = pd.read_csv(zones_url)
zones_gpd = gpd.GeoDataFrame(zones_pd, geometry=gpd.GeoSeries.from_wkt(zones_pd["the_geom"]), crs="EPSG:4326")
bronze_aq = spark.read.table("Bronze_Lakehouse.dbo.openaq_raw")
air_locations = (bronze_aq.select("location_id", "latitude", "longitude")
.dropDuplicates().toPandas().dropna(subset=["latitude", "longitude"]))

air_points = gpd.GeoDataFrame(air_locations,
geometry=gpd.points_from_xy(air_locations["longitude"],air_locations["latitude"]), crs="EPSG:4326")
zone_premapping = gpd.sjoin(air_points, zones_gpd[["locationid", "zone", "geometry"]],how="left", predicate="within")
zone_mapping = (pd.DataFrame(zone_premapping[["location_id", "locationid", "zone"]])
.drop_duplicates(subset=["location_id"]))
zone_mapping_spark = spark.createDataFrame(zone_mapping)

silver_aq = (
bronze_aq
.withColumn("timestamp_utc", to_timestamp("datetime_utc"))
.withColumn("timestamp_local",from_utc_timestamp(col("timestamp_utc"), "America/New_York"))
.withColumn("date_key", to_date("timestamp_local"))
.withColumn("pollutant_value", col("value").cast("double"))
.withColumn("pollutant_type", col("parameter"))

.filter(col("pollutant_value").isNotNull())
.filter(col("pollutant_value").between(0, 500))
.join(zone_mapping_spark, on="location_id", how="inner")
.withColumn("locationid", col("locationid").cast("int"))
.select("date_key", "pollutant_type", "pollutant_value", "locationid", "zone")
)

print(f"Total rows: {silver_aq.count():,}")
(silver_aq.write.mode("overwrite").option("overwriteSchema", "true")
.format("delta").saveAsTable("Silver_Lakehouse.dbo.openaq_silver"))

In [ ]:
from pyspark.sql.functions import col
bronze_gdp = spark.read.table("Bronze_Lakehouse.dbo.gdp_raw")

silver_gdp = (
    bronze_gdp
    .withColumn("gdp_usd", col("gdp_usd").cast("double"))
    .withColumn("year", col("year").cast("int"))
    .select("country_name", "year", "gdp_usd")
)

(
    silver_gdp.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("Silver_Lakehouse.dbo.gdp_silver")
)
print(f"Total rows: {silver_gdp.count():,}")

In [ ]:
from pyspark.sql.functions import col, to_date, sequence, explode, min, max, last
from pyspark.sql.window import Window

bronze_fx = spark.read.table("Bronze_Lakehouse.dbo.fx_raw")
fx_cleaned = (
bronze_fx
.withColumn("date_key", col("date_string").cast("date"))
.withColumn("eur_to_usd", col("eur_to_usd").cast("double"))
.filter(col("date_key").isNotNull())
.filter(col("eur_to_usd").isNotNull())
.select("date_key", "eur_to_usd")
)

date_range = fx_cleaned.agg(min("date_key").alias("min_date"),max("date_key").alias("max_date"))
calendar = date_range.select(explode(sequence(col("min_date"), col("max_date"))).alias("date_key"))
fx_daily = calendar.join(fx_cleaned, on="date_key", how="left")
window_spec = Window.orderBy("date_key").rowsBetween(Window.unboundedPreceding, Window.currentRow)

silver_fx = (
fx_daily
.withColumn("eur_to_usd",last("eur_to_usd", ignorenulls=True).over(window_spec))
.orderBy("date_key")
)

(silver_fx.write
.mode("overwrite")
.option("overwriteSchema", "true")
.format("delta")
.saveAsTable("Silver_Lakehouse.dbo.fx_silver"))
print(f"Total rows: {silver_fx.count():,}")